# Módulo 5 · Clase 4 — FastAPI avanzado

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | FastAPI + SQLAlchemy: integración completa |
| 2 | Modelos ORM y esquemas Pydantic separados |
| 3 | Sesiones de base de datos con `Depends` |
| 4 | CRUD real con persistencia |
| 5 | Manejo de errores robusto |
| 6 | Relaciones entre modelos |
| 7 | Background tasks |

> La diferencia clave con la clase anterior: los datos ahora **sobreviven al reinicio del servidor** porque los guardamos en SQLite.

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'sqlalchemy', 'requests']:
    try:
        __import__(pkg.split('[')[0])
        print(f"{pkg.split('[')[0]}: ya instalado")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: instalado")

---
## 1. El patrón completo: FastAPI + SQLAlchemy

Cuando combinas FastAPI con una base de datos, usas tres capas bien separadas:

```
┌─────────────────────────────────────────────────────┐
│  CAPA 1: Base de datos                              │
│  engine, SessionLocal, Base = declarative_base()    │
├─────────────────────────────────────────────────────┤
│  CAPA 2: Modelos ORM (models.py)                    │
│  class Libro(Base):  __tablename__ = "libros"       │
│      id = Column(Integer, primary_key=True)         │
├─────────────────────────────────────────────────────┤
│  CAPA 3: Esquemas Pydantic (schemas.py)             │
│  class LibroCreate(BaseModel): ...                  │
│  class LibroResponse(BaseModel): ...                │
├─────────────────────────────────────────────────────┤
│  CAPA 4: Endpoints FastAPI (main.py)                │
│  @app.post("/libros")                               │
│  def crear(libro: LibroCreate, db: Session = ...):  │
└─────────────────────────────────────────────────────┘

¿Por qué separar modelos ORM de esquemas Pydantic?
- El modelo ORM define la tabla en BD
- El esquema Pydantic define qué entra/sale por la API
- No siempre coinciden: la password entra pero nunca sale
```

---
## 2. Configuración de base de datos

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean, ForeignKey, DateTime, Text
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker
from datetime import datetime

# ── Motor de base de datos ───────────────────────────────────
DATABASE_URL = "sqlite:///./biblioteca.db"
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}  # necesario para SQLite con FastAPI
)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


# ── Modelos ORM ──────────────────────────────────────────────
class Autor(Base):
    __tablename__ = "autores"

    id         = Column(Integer, primary_key=True, autoincrement=True)
    nombre     = Column(String(100), nullable=False)
    pais       = Column(String(100), nullable=True)
    nacimiento = Column(Integer, nullable=True)

    libros = relationship("Libro", back_populates="autor", cascade="all, delete-orphan")


class Libro(Base):
    __tablename__ = "libros"

    id         = Column(Integer, primary_key=True, autoincrement=True)
    titulo     = Column(String(200), nullable=False)
    isbn       = Column(String(20),  unique=True, nullable=True)
    anio       = Column(Integer,     nullable=True)
    precio     = Column(Float,       nullable=True)
    disponible = Column(Boolean,     default=True)
    resumen    = Column(Text,        nullable=True)
    creado_en  = Column(DateTime,    default=datetime.utcnow)
    autor_id   = Column(Integer,     ForeignKey("autores.id", ondelete="CASCADE"))

    autor = relationship("Autor", back_populates="libros")


# Crear tablas
Base.metadata.drop_all(engine)   # limpiar en cada ejecución del notebook
Base.metadata.create_all(engine)
print("Base de datos lista. Tablas:", engine.dialect.get_table_names(engine.connect()))

---
## 3. Sesiones con `Depends`

FastAPI usa **inyección de dependencias** para gestionar la sesión de BD por request:

```python
def get_db():
    db = SessionLocal()
    try:
        yield db       # entrega la sesión al endpoint
    finally:
        db.close()     # siempre se cierra, aunque haya error

@app.get("/libros")
def listar(db: Session = Depends(get_db)):  # FastAPI llama get_db automáticamente
    return db.query(Libro).all()
```

In [ ]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional

# ── Esquemas Pydantic (entrada/salida API) ───────────────────

class AutorCreate(BaseModel):
    nombre:     str            = Field(min_length=1, max_length=100)
    pais:       Optional[str]  = None
    nacimiento: Optional[int]  = None

class AutorResponse(AutorCreate):
    id: int
    model_config = ConfigDict(from_attributes=True)  # leer desde modelo ORM

class AutorConLibros(AutorResponse):
    libros: list["LibroResponse"] = []


class LibroCreate(BaseModel):
    titulo:     str            = Field(min_length=1, max_length=200)
    isbn:       Optional[str]  = None
    anio:       Optional[int]  = Field(None, ge=1000, le=2100)
    precio:     Optional[float]= Field(None, gt=0)
    disponible: bool           = True
    resumen:    Optional[str]  = None
    autor_id:   int

class LibroUpdate(BaseModel):
    titulo:     Optional[str]  = Field(None, min_length=1)
    precio:     Optional[float]= Field(None, gt=0)
    disponible: Optional[bool] = None
    resumen:    Optional[str]  = None

class LibroResponse(BaseModel):
    id:         int
    titulo:     str
    isbn:       Optional[str]
    anio:       Optional[int]
    precio:     Optional[float]
    disponible: bool
    autor_id:   int
    model_config = ConfigDict(from_attributes=True)

class LibroConAutor(LibroResponse):
    autor: Optional[AutorResponse] = None

AutorConLibros.model_rebuild()  # necesario por referencia circular
print("Esquemas listos")

---
## 4. API completa con CRUD

In [ ]:
from fastapi import FastAPI, HTTPException, Depends, Query, BackgroundTasks
from sqlalchemy.orm import Session
import uvicorn, threading, time, requests

app = FastAPI(title="API Biblioteca", version="1.0.0")

# Dependencia de sesión
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


# ════════════════════════════════════════════════════════════
#  AUTORES
# ════════════════════════════════════════════════════════════

@app.get("/autores", response_model=list[AutorResponse], tags=["Autores"])
def listar_autores(
    skip:  int = Query(0, ge=0),
    limit: int = Query(20, ge=1, le=100),
    db:    Session = Depends(get_db)
):
    return db.query(Autor).offset(skip).limit(limit).all()


@app.get("/autores/{autor_id}", response_model=AutorConLibros, tags=["Autores"])
def obtener_autor(autor_id: int, db: Session = Depends(get_db)):
    autor = db.get(Autor, autor_id)
    if not autor:
        raise HTTPException(status_code=404, detail="Autor no encontrado")
    return autor


@app.post("/autores", response_model=AutorResponse, status_code=201, tags=["Autores"])
def crear_autor(datos: AutorCreate, db: Session = Depends(get_db)):
    autor = Autor(**datos.model_dump())
    db.add(autor)
    db.commit()
    db.refresh(autor)
    return autor


@app.delete("/autores/{autor_id}", status_code=204, tags=["Autores"])
def eliminar_autor(autor_id: int, db: Session = Depends(get_db)):
    autor = db.get(Autor, autor_id)
    if not autor:
        raise HTTPException(status_code=404, detail="Autor no encontrado")
    db.delete(autor)
    db.commit()


# ════════════════════════════════════════════════════════════
#  LIBROS
# ════════════════════════════════════════════════════════════

@app.get("/libros", response_model=list[LibroConAutor], tags=["Libros"])
def listar_libros(
    skip:       int            = Query(0, ge=0),
    limit:      int            = Query(20, le=100),
    disponible: Optional[bool] = None,
    autor_id:   Optional[int]  = None,
    q:          Optional[str]  = None,
    db:         Session        = Depends(get_db)
):
    consulta = db.query(Libro)
    if disponible is not None:
        consulta = consulta.filter(Libro.disponible == disponible)
    if autor_id:
        consulta = consulta.filter(Libro.autor_id == autor_id)
    if q:
        consulta = consulta.filter(Libro.titulo.ilike(f"%{q}%"))
    return consulta.offset(skip).limit(limit).all()


@app.get("/libros/{libro_id}", response_model=LibroConAutor, tags=["Libros"])
def obtener_libro(libro_id: int, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if not libro:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    return libro


@app.post("/libros", response_model=LibroConAutor, status_code=201, tags=["Libros"])
def crear_libro(datos: LibroCreate, db: Session = Depends(get_db)):
    # Verificar que el autor existe
    if not db.get(Autor, datos.autor_id):
        raise HTTPException(status_code=404, detail="Autor no encontrado")

    # Verificar ISBN único si se proporciona
    if datos.isbn:
        existente = db.query(Libro).filter(Libro.isbn == datos.isbn).first()
        if existente:
            raise HTTPException(status_code=409, detail=f"ISBN {datos.isbn} ya registrado")

    libro = Libro(**datos.model_dump())
    db.add(libro)
    db.commit()
    db.refresh(libro)
    return libro


@app.patch("/libros/{libro_id}", response_model=LibroConAutor, tags=["Libros"])
def actualizar_libro(libro_id: int, cambios: LibroUpdate, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if not libro:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    for campo, valor in cambios.model_dump(exclude_none=True).items():
        setattr(libro, campo, valor)
    db.commit()
    db.refresh(libro)
    return libro


@app.delete("/libros/{libro_id}", status_code=204, tags=["Libros"])
def eliminar_libro(libro_id: int, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if not libro:
        raise HTTPException(status_code=404, detail="Libro no encontrado")
    db.delete(libro)
    db.commit()


# ════════════════════════════════════════════════════════════
#  ESTADÍSTICAS
# ════════════════════════════════════════════════════════════

@app.get("/estadisticas", tags=["Info"])
def estadisticas(db: Session = Depends(get_db)):
    from sqlalchemy import func
    total_libros    = db.query(func.count(Libro.id)).scalar()
    libros_dispon   = db.query(func.count(Libro.id)).filter(Libro.disponible).scalar()
    total_autores   = db.query(func.count(Autor.id)).scalar()
    precio_medio    = db.query(func.avg(Libro.precio)).scalar()
    return {
        "total_libros":     total_libros,
        "libros_disponibles": libros_dispon,
        "total_autores":    total_autores,
        "precio_medio":     round(precio_medio or 0, 2)
    }


# Arrancar servidor
cfg = uvicorn.Config(app, host="127.0.0.1", port=8010, log_level="warning")
srv = uvicorn.Server(cfg)
threading.Thread(target=srv.run, daemon=True).start()
time.sleep(1.5)
print("Servidor en http://127.0.0.1:8010")
print("Docs:        http://127.0.0.1:8010/docs")

In [ ]:
# ── Demo completo ────────────────────────────────────────────
B = "http://127.0.0.1:8010"
s = requests.Session()
s.headers.update({"Content-Type": "application/json"})

print("=== Crear autores ===")
a1 = s.post(f"{B}/autores", json={"nombre": "Gabriel García Márquez", "pais": "Colombia", "nacimiento": 1927}).json()
a2 = s.post(f"{B}/autores", json={"nombre": "Isabel Allende", "pais": "Chile", "nacimiento": 1942}).json()
print(f"  Autores: {a1['id']}. {a1['nombre']}, {a2['id']}. {a2['nombre']}")

print("\n=== Crear libros ===")
l1 = s.post(f"{B}/libros", json={"titulo": "Cien años de soledad", "isbn": "978-0307474728",
    "anio": 1967, "precio": 14.95, "autor_id": a1['id']}).json()
l2 = s.post(f"{B}/libros", json={"titulo": "El amor en los tiempos del cólera",
    "anio": 1985, "precio": 12.50, "autor_id": a1['id']}).json()
l3 = s.post(f"{B}/libros", json={"titulo": "La casa de los espíritus",
    "anio": 1982, "precio": 13.00, "disponible": False, "autor_id": a2['id']}).json()
print(f"  Libros creados: {l1['titulo']}, {l2['titulo']}, {l3['titulo']}")

print("\n=== ISBN duplicado (409) ===")
r = s.post(f"{B}/libros", json={"titulo": "Copia", "isbn": "978-0307474728", "autor_id": a1['id']})
print(f"  status={r.status_code}, detail={r.json()['detail']}")

print("\n=== Autor con sus libros ===")
autor_detalle = s.get(f"{B}/autores/{a1['id']}").json()
print(f"  {autor_detalle['nombre']}: {[l['titulo'] for l in autor_detalle['libros']]}")

print("\n=== Filtrar disponibles ===")
disponibles = s.get(f"{B}/libros", params={"disponible": True}).json()
print(f"  Disponibles: {[l['titulo'] for l in disponibles]}")

print("\n=== Actualizar precio ===")
updated = s.patch(f"{B}/libros/{l1['id']}", json={"precio": 16.99}).json()
print(f"  {updated['titulo']}: {updated['precio']}€")

print("\n=== Estadísticas ===")
print(s.get(f"{B}/estadisticas").json())

---
## 7. Background Tasks

FastAPI permite ejecutar tareas en segundo plano después de devolver la respuesta:

```python
@app.post("/libros")
def crear(datos: LibroCreate, background_tasks: BackgroundTasks, db: Session = Depends(get_db)):
    libro = Libro(**datos.model_dump())
    db.add(libro); db.commit()

    # Se ejecuta DESPUÉS de devolver la respuesta
    background_tasks.add_task(enviar_notificacion, libro.titulo)

    return libro
```

Útil para: envío de emails, notificaciones, logs asíncronos, cálculos pesados.

---
## Práctica

### Ejercicio 1 — Añadir categorías

**Enunciado:** Añade un modelo `Categoria` a la API de la biblioteca:

- Tabla `categorias`: id, nombre (único), descripcion (opcional)
- Relaciona `Libro` con `Categoria` (1:N — un libro tiene una categoría)
- Añade `categoria_id` como campo opcional en `LibroCreate`
- Endpoint `GET /categorias` y `POST /categorias`
- Al listar libros, incluye el nombre de la categoría si tiene

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean, ForeignKey, Text
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker
from fastapi import FastAPI, HTTPException, Depends
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional

engine2 = create_engine("sqlite:///./bib2.db", connect_args={"check_same_thread": False})
SessionLocal2 = sessionmaker(bind=engine2)
Base2 = declarative_base()

class Categoria(Base2):
    __tablename__ = "categorias"
    id          = Column(Integer, primary_key=True)
    nombre      = Column(String(100), nullable=False, unique=True)
    descripcion = Column(Text, nullable=True)
    libros      = relationship("Libro2", back_populates="categoria")

class Libro2(Base2):
    __tablename__ = "libros2"
    id           = Column(Integer, primary_key=True)
    titulo       = Column(String(200), nullable=False)
    precio       = Column(Float, nullable=True)
    categoria_id = Column(Integer, ForeignKey("categorias.id"), nullable=True)
    categoria    = relationship("Categoria", back_populates="libros")

Base2.metadata.create_all(engine2)

class CategoriaCreate(BaseModel):
    nombre:      str           = Field(min_length=1)
    descripcion: Optional[str] = None

class CategoriaResponse(CategoriaCreate):
    id: int
    model_config = ConfigDict(from_attributes=True)

class Libro2Create(BaseModel):
    titulo:       str           = Field(min_length=1)
    precio:       Optional[float] = None
    categoria_id: Optional[int] = None

class Libro2Response(Libro2Create):
    id:        int
    categoria: Optional[CategoriaResponse] = None
    model_config = ConfigDict(from_attributes=True)

app2 = FastAPI()

def get_db2():
    db = SessionLocal2()
    try:   yield db
    finally: db.close()

@app2.get("/categorias", response_model=list[CategoriaResponse])
def listar_cats(db: Session = Depends(get_db2)):
    return db.query(Categoria).all()

@app2.post("/categorias", response_model=CategoriaResponse, status_code=201)
def crear_cat(datos: CategoriaCreate, db: Session = Depends(get_db2)):
    existente = db.query(Categoria).filter(Categoria.nombre == datos.nombre).first()
    if existente: raise HTTPException(409, "Categoría ya existe")
    cat = Categoria(**datos.model_dump())
    db.add(cat); db.commit(); db.refresh(cat)
    return cat

@app2.get("/libros", response_model=list[Libro2Response])
def listar_libros(db: Session = Depends(get_db2)):
    return db.query(Libro2).all()

@app2.post("/libros", response_model=Libro2Response, status_code=201)
def crear_libro(datos: Libro2Create, db: Session = Depends(get_db2)):
    if datos.categoria_id and not db.get(Categoria, datos.categoria_id):
        raise HTTPException(404, "Categoría no encontrada")
    libro = Libro2(**datos.model_dump())
    db.add(libro); db.commit(); db.refresh(libro)
    return libro
```
</details>

---
### Ejercicio 2 — Búsqueda avanzada

**Enunciado:** Añade un endpoint `GET /buscar` que permita buscar libros con múltiples criterios combinados:

- `q` — texto en título o resumen (case insensitive)
- `anio_min`, `anio_max` — rango de año de publicación
- `precio_max` — precio máximo
- `disponible` — solo disponibles
- `ordenar_por` — `titulo`, `precio`, `anio` (default: `titulo`)
- `orden` — `asc` o `desc` (default: `asc`)

Devuelve también el total de resultados antes de paginar.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from fastapi import Query
from sqlalchemy import asc, desc
from typing import Optional, Literal

@app.get("/buscar", tags=["Libros"])
def buscar(
    q:          Optional[str]   = None,
    anio_min:   Optional[int]   = None,
    anio_max:   Optional[int]   = None,
    precio_max: Optional[float] = Query(None, gt=0),
    disponible: Optional[bool]  = None,
    ordenar_por: Literal["titulo", "precio", "anio"] = "titulo",
    orden:       Literal["asc", "desc"]              = "asc",
    skip:        int = 0,
    limit:       int = Query(20, le=100),
    db:          Session = Depends(get_db)
):
    consulta = db.query(Libro)

    if q:
        consulta = consulta.filter(
            Libro.titulo.ilike(f"%{q}%") | Libro.resumen.ilike(f"%{q}%")
        )
    if anio_min:   consulta = consulta.filter(Libro.anio >= anio_min)
    if anio_max:   consulta = consulta.filter(Libro.anio <= anio_max)
    if precio_max: consulta = consulta.filter(Libro.precio <= precio_max)
    if disponible is not None: consulta = consulta.filter(Libro.disponible == disponible)

    col_orden = getattr(Libro, ordenar_por)
    consulta = consulta.order_by(asc(col_orden) if orden == "asc" else desc(col_orden))

    total   = consulta.count()
    items   = consulta.offset(skip).limit(limit).all()

    return {
        "total": total, "skip": skip, "limit": limit,
        "items": [LibroConAutor.model_validate(i) for i in items]
    }
```
</details>

---
### Ejercicio 3 — Validación de negocio

**Enunciado:** Añade estas reglas de negocio a la API de libros:

1. El año de publicación no puede ser futuro.
2. Si el libro no está disponible, su precio no puede modificarse.
3. Al eliminar un autor, verificar primero si tiene libros y devolver 409 con la cantidad.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
from datetime import datetime

# En crear_libro:
anio_actual = datetime.now().year
if datos.anio and datos.anio > anio_actual:
    raise HTTPException(status_code=422, detail=f"El año no puede ser mayor a {anio_actual}")

# En actualizar_libro:
if not libro.disponible and cambios.precio is not None:
    raise HTTPException(status_code=409, detail="No se puede modificar el precio de un libro no disponible")

# En eliminar_autor:
n_libros = db.query(Libro).filter(Libro.autor_id == autor_id).count()
if n_libros > 0:
    raise HTTPException(
        status_code=409,
        detail=f"No se puede eliminar: el autor tiene {n_libros} libro(s). Elimínalos primero."
    )
```
</details>

---
### Ejercicio 4 — Préstamos

**Enunciado:** Añade un sistema de préstamos a la biblioteca:

- Modelo `Prestamo`: id, libro_id (FK), usuario (str), fecha_inicio, fecha_devolucion (nullable), activo (bool)
- `POST /libros/{id}/prestar` — presta un libro (solo si está disponible)
  - Marca el libro como no disponible
  - Crea el registro de préstamo
- `POST /prestamos/{id}/devolver` — devuelve el libro
  - Marca el libro como disponible
  - Registra la fecha de devolución
- `GET /prestamos/activos` — lista los préstamos en curso

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from sqlalchemy import Column, Integer, String, Boolean, ForeignKey, DateTime
from datetime import datetime

class Prestamo(Base):
    __tablename__ = "prestamos"
    id               = Column(Integer, primary_key=True)
    libro_id         = Column(Integer, ForeignKey("libros.id"), nullable=False)
    usuario          = Column(String(100), nullable=False)
    fecha_inicio     = Column(DateTime, default=datetime.utcnow)
    fecha_devolucion = Column(DateTime, nullable=True)
    activo           = Column(Boolean, default=True)

# No olvides: Base.metadata.create_all(engine)

class PrestamoCreate(BaseModel):
    usuario: str = Field(min_length=1)

class PrestamoResponse(BaseModel):
    id: int; libro_id: int; usuario: str
    fecha_inicio: datetime; fecha_devolucion: Optional[datetime]; activo: bool
    model_config = ConfigDict(from_attributes=True)

@app.post("/libros/{libro_id}/prestar", response_model=PrestamoResponse, status_code=201)
def prestar(libro_id: int, datos: PrestamoCreate, db: Session = Depends(get_db)):
    libro = db.get(Libro, libro_id)
    if not libro:           raise HTTPException(404, "Libro no encontrado")
    if not libro.disponible: raise HTTPException(409, "El libro ya está prestado")

    libro.disponible = False
    prestamo = Prestamo(libro_id=libro_id, usuario=datos.usuario)
    db.add(prestamo); db.commit(); db.refresh(prestamo)
    return prestamo

@app.post("/prestamos/{prest_id}/devolver", response_model=PrestamoResponse)
def devolver(prest_id: int, db: Session = Depends(get_db)):
    prestamo = db.get(Prestamo, prest_id)
    if not prestamo:       raise HTTPException(404, "Préstamo no encontrado")
    if not prestamo.activo: raise HTTPException(409, "Este préstamo ya fue devuelto")

    prestamo.activo           = False
    prestamo.fecha_devolucion = datetime.utcnow()
    libro = db.get(Libro, prestamo.libro_id)
    libro.disponible = True
    db.commit(); db.refresh(prestamo)
    return prestamo

@app.get("/prestamos/activos", response_model=list[PrestamoResponse])
def prestamos_activos(db: Session = Depends(get_db)):
    return db.query(Prestamo).filter(Prestamo.activo).all()
```
</details>

---
### Ejercicio 5 (reto) — API de recetas

**Enunciado:** Construye desde cero una API de recetas de cocina con persistencia:

**Modelos:**
- `Ingrediente`: id, nombre (único), unidad ("g", "ml", "ud", "taza")
- `Receta`: id, nombre, descripcion, tiempo_min (int), dificultad ("facil", "media", "dificil")
- `RecetaIngrediente`: receta_id, ingrediente_id, cantidad (float) — tabla intermedia N:M

**Endpoints mínimos:**
- CRUD de ingredientes
- CRUD de recetas
- `POST /recetas/{id}/ingredientes` — añade un ingrediente a la receta
- `GET /recetas/{id}` — incluye la lista de ingredientes con cantidades
- `GET /recetas/buscar?ingrediente=harina` — recetas que contienen un ingrediente

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Enum as SQLEnum
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker
from fastapi import FastAPI, HTTPException, Depends
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Literal
import enum, uvicorn, threading, time, requests

engine_r = create_engine("sqlite:///./recetas.db", connect_args={"check_same_thread": False})
SessionR  = sessionmaker(bind=engine_r)
BaseR     = declarative_base()

class Unidad(str, enum.Enum):
    g = "g"; ml = "ml"; ud = "ud"; taza = "taza"

class Dificultad(str, enum.Enum):
    facil = "facil"; media = "media"; dificil = "dificil"

class Ingrediente(BaseR):
    __tablename__ = "ingredientes"
    id     = Column(Integer, primary_key=True)
    nombre = Column(String(100), nullable=False, unique=True)
    unidad = Column(SQLEnum(Unidad), nullable=False)

class Receta(BaseR):
    __tablename__ = "recetas"
    id          = Column(Integer, primary_key=True)
    nombre      = Column(String(200), nullable=False)
    descripcion = Column(String(500), nullable=True)
    tiempo_min  = Column(Integer, nullable=True)
    dificultad  = Column(SQLEnum(Dificultad), default=Dificultad.media)
    ingredientes = relationship("RecetaIngrediente", back_populates="receta", cascade="all, delete-orphan")

class RecetaIngrediente(BaseR):
    __tablename__ = "receta_ingrediente"
    receta_id     = Column(Integer, ForeignKey("recetas.id"), primary_key=True)
    ingrediente_id= Column(Integer, ForeignKey("ingredientes.id"), primary_key=True)
    cantidad      = Column(Float, nullable=False)
    receta        = relationship("Receta", back_populates="ingredientes")
    ingrediente   = relationship("Ingrediente")

BaseR.metadata.create_all(engine_r)

# Esquemas
class IngCreate(BaseModel):
    nombre: str = Field(min_length=1); unidad: Unidad
class IngResponse(IngCreate):
    id: int; model_config = ConfigDict(from_attributes=True)

class RecetaCreate(BaseModel):
    nombre: str = Field(min_length=1); descripcion: Optional[str] = None
    tiempo_min: Optional[int] = None; dificultad: Dificultad = Dificultad.media
class RecetaIngCreate(BaseModel):
    ingrediente_id: int; cantidad: float = Field(gt=0)
class RecetaIngResponse(BaseModel):
    ingrediente: IngResponse; cantidad: float
    model_config = ConfigDict(from_attributes=True)
class RecetaResponse(RecetaCreate):
    id: int; ingredientes: list[RecetaIngResponse] = []
    model_config = ConfigDict(from_attributes=True)

app_r = FastAPI(title="API Recetas")
def get_db(): db = SessionR(); yield db; db.close()

@app_r.get("/ingredientes", response_model=list[IngResponse])
def listar_ing(db: Session = Depends(get_db)): return db.query(Ingrediente).all()

@app_r.post("/ingredientes", response_model=IngResponse, status_code=201)
def crear_ing(datos: IngCreate, db: Session = Depends(get_db)):
    if db.query(Ingrediente).filter(Ingrediente.nombre == datos.nombre).first():
        raise HTTPException(409, "Ingrediente ya existe")
    ing = Ingrediente(**datos.model_dump()); db.add(ing); db.commit(); db.refresh(ing); return ing

@app_r.get("/recetas/buscar", response_model=list[RecetaResponse])
def buscar_recetas(ingrediente: str, db: Session = Depends(get_db)):
    return db.query(Receta).join(RecetaIngrediente).join(Ingrediente).filter(
        Ingrediente.nombre.ilike(f"%{ingrediente}%")
    ).all()

@app_r.get("/recetas", response_model=list[RecetaResponse])
def listar_rec(db: Session = Depends(get_db)): return db.query(Receta).all()

@app_r.post("/recetas", response_model=RecetaResponse, status_code=201)
def crear_rec(datos: RecetaCreate, db: Session = Depends(get_db)):
    rec = Receta(**datos.model_dump()); db.add(rec); db.commit(); db.refresh(rec); return rec

@app_r.get("/recetas/{id}", response_model=RecetaResponse)
def obtener_rec(id: int, db: Session = Depends(get_db)):
    rec = db.get(Receta, id)
    if not rec: raise HTTPException(404, "Receta no encontrada")
    return rec

@app_r.post("/recetas/{id}/ingredientes", status_code=201)
def añadir_ing(id: int, datos: RecetaIngCreate, db: Session = Depends(get_db)):
    if not db.get(Receta, id): raise HTTPException(404, "Receta no encontrada")
    if not db.get(Ingrediente, datos.ingrediente_id): raise HTTPException(404, "Ingrediente no encontrado")
    ri = RecetaIngrediente(receta_id=id, **datos.model_dump())
    db.merge(ri); db.commit()
    return {"mensaje": "Ingrediente añadido"}
```
</details>

---
## Resumen

```python
# 1. Configurar BD
engine       = create_engine("sqlite:///./app.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)
Base         = declarative_base()

# 2. Modelo ORM
class Libro(Base):
    __tablename__ = "libros"
    id    = Column(Integer, primary_key=True)
    titulo = Column(String, nullable=False)

# 3. Esquema Pydantic (separado del ORM)
class LibroResponse(BaseModel):
    id: int; titulo: str
    model_config = ConfigDict(from_attributes=True)

# 4. Dependencia de sesión
def get_db():
    db = SessionLocal()
    try:   yield db
    finally: db.close()

# 5. Endpoint con BD
@app.get("/libros", response_model=list[LibroResponse])
def listar(db: Session = Depends(get_db)):
    return db.query(Libro).all()

@app.post("/libros", status_code=201)
def crear(datos: LibroCreate, db: Session = Depends(get_db)):
    libro = Libro(**datos.model_dump())
    db.add(libro); db.commit(); db.refresh(libro)
    return libro
```

## ¿Qué viene en la próxima clase?

En la **Clase 5** añadimos autenticación real: API keys, OAuth2 con password flow y tokens JWT para proteger los endpoints.